# Import Libraries

In [13]:
import random
import csv
import matplotlib.pyplot as plt

# Parameter Values

In [14]:
# CONFIGURATIONS

GRID_SIZE = 50
TIME_STEPS = 200

INITIAL_PREY = 40
INITIAL_PREDATORS = 10

MAX_PREY = 5000
MAX_PREDATORS = 800

# FOOD SYSTEM
FOOD_MAX = 5
FOOD_REGROWTH = 1

In [15]:

# AGENTS
class Prey:
    def __init__(self, x, y, energy=6):
        self.x = x
        self.y = y
        self.energy = energy

class Predator:
    def __init__(self, x, y, energy=8):
        self.x = x
        self.y = y
        self.energy = energy

# WORLD

class World:
    def __init__(self, params):
        self.params = params
        self.prey = []
        self.predators = []

        # FOOD GRID
        self.food = [[random.randint(1, FOOD_MAX) for _ in range(GRID_SIZE)]
                     for _ in range(GRID_SIZE)]

    def wrap(self, x, y):
        return x % GRID_SIZE, y % GRID_SIZE

    # SENSING

    def sense_predators(self, prey):
        d = self.params["prey_sense"]
        return [p for p in self.predators
                if abs(p.x - prey.x) <= d and abs(p.y - prey.y) <= d]

    def sense_prey(self, predator):
        d = self.params["predator_sense"]
        return [pr for pr in self.prey
                if abs(pr.x - predator.x) <= d and abs(pr.y - predator.y) <= d]

    # MOVEMENT

    def move_prey(self):
        for prey in self.prey:
            predators = self.sense_predators(prey)

            moves = [(dx, dy)
                     for dx in range(-self.params["prey_move"], self.params["prey_move"] + 1)
                     for dy in range(-self.params["prey_move"], self.params["prey_move"] + 1)
                     if not (dx == 0 and dy == 0)]

            if predators:
                nearest = min(predators,
                              key=lambda p: abs(p.x - prey.x) + abs(p.y - prey.y))

                best_move = max(
                    moves,
                    key=lambda m: abs((prey.x + m[0]) - nearest.x) +
                                  abs((prey.y + m[1]) - nearest.y)
                )
            else:
                # move toward food (smart behavior)
                best_move = max(
                    moves,
                    key=lambda m: self.food[(prey.x + m[0]) % GRID_SIZE]
                                             [(prey.y + m[1]) % GRID_SIZE]
                )

            prey.x, prey.y = self.wrap(prey.x + best_move[0],
                                       prey.y + best_move[1])

            prey.energy -= 1  # movement cost

    def move_predators(self):
        for predator in self.predators:
            prey_list = self.sense_prey(predator)

            moves = [(dx, dy)
                     for dx in range(-self.params["predator_move"], self.params["predator_move"] + 1)
                     for dy in range(-self.params["predator_move"], self.params["predator_move"] + 1)
                     if not (dx == 0 and dy == 0)]

            if prey_list:
                target = min(prey_list,
                             key=lambda pr: abs(pr.x - predator.x) +
                                            abs(pr.y - predator.y))

                best_move = min(
                    moves,
                    key=lambda m: abs((predator.x + m[0]) - target.x) +
                                  abs((predator.y + m[1]) - target.y)
                )
            else:
                best_move = random.choice(moves)

            predator.x, predator.y = self.wrap(predator.x + best_move[0],
                                               predator.y + best_move[1])

            predator.energy -= 1

    # INTERACTION

    def eat(self):
        for predator in self.predators:
            same_cell = [p for p in self.prey
                         if p.x == predator.x and p.y == predator.y]

            if same_cell:
                victim = random.choice(same_cell)
                self.prey.remove(victim)
                predator.energy += 12

    # FOOD CONSUMPTION

    def prey_eat_food(self):
        for prey in self.prey:
            if self.food[prey.x][prey.y] > 0:
                prey.energy += min(3, self.food[prey.x][prey.y])
                self.food[prey.x][prey.y] = 0

    def regrow_food(self):
        for i in range(GRID_SIZE):
            for j in range(GRID_SIZE):
                if self.food[i][j] < FOOD_MAX:
                    self.food[i][j] += FOOD_REGROWTH

    # REPRODUCTION

    def reproduce(self):
        new_prey = []

        for prey in self.prey:
            if prey.energy >= 10 and random.random() < self.params["prey_reproduce"]:
                dx, dy = random.randint(-1, 1), random.randint(-1, 1)
                nx, ny = self.wrap(prey.x + dx, prey.y + dy)
                new_prey.append(Prey(nx, ny, energy=prey.energy // 2))
                prey.energy //= 2

        self.prey.extend(new_prey)

        new_predators = []

        for predator in self.predators:
            if predator.energy >= self.params["predator_reproduce_energy"]:
                dx, dy = random.randint(-1, 1), random.randint(-1, 1)
                nx, ny = self.wrap(predator.x + dx, predator.y + dy)
                new_predators.append(Predator(nx, ny, energy=predator.energy // 2))
                predator.energy //= 2

        self.predators.extend(new_predators)

    # CLEANUP

    def remove_dead(self):
        self.prey = [p for p in self.prey if p.energy > 0]
        self.predators = [p for p in self.predators if p.energy > 0]

# SIMULATION

def run_simulation(params, run_id):
    world = World(params)

    for _ in range(INITIAL_PREY):
        world.prey.append(Prey(random.randint(0, 49), random.randint(0, 49)))

    for _ in range(INITIAL_PREDATORS):
        world.predators.append(Predator(random.randint(0, 49), random.randint(0, 49)))

    prey_counts = []
    predator_counts = []

    for step in range(TIME_STEPS):
        world.move_prey()
        world.move_predators()

        world.prey_eat_food()   # 🔥 NEW
        world.eat()

        world.remove_dead()
        world.reproduce()

        world.regrow_food()     # 🔥 NEW

        prey_counts.append(len(world.prey))
        predator_counts.append(len(world.predators))

    # Saving CSV files
    with open(f"task3_run_{run_id}.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Step", "Prey", "Predators"])
        for i in range(TIME_STEPS):
            writer.writerow([i, prey_counts[i], predator_counts[i]])

    # Plots
    plt.figure()
    plt.plot(prey_counts, label="Prey")
    plt.plot(predator_counts, label="Predators")

    plt.xlabel("Time Step")          # Horizontal axis
    plt.ylabel("Population Size")    # Vertical axis
    plt.legend()
    plt.title(f"Task 3 Run {run_id}")
    plt.savefig(f"task3_run_{run_id}.png")
    plt.close()

# Parameter Settings

param_set_1 = {
    "prey_reproduce": 0.08,
    "predator_reproduce_energy": 10,
    "prey_sense": 3,
    "predator_sense": 3,
    "prey_move": 1,
    "predator_move": 2
}

param_set_2 = {
    "prey_reproduce": 0.07,
    "predator_reproduce_energy": 12,
    "prey_sense": 2,
    "predator_sense": 5,
    "prey_move": 1,
    "predator_move": 2
}


In [16]:
# Running the file

run_id = 1

for params in [param_set_1, param_set_2]:
    for _ in range(3):
        print(f"Running Task 3 simulation {run_id}")
        run_simulation(params, run_id)
        run_id += 1

print("Task 3 complete!")

Running Task 3 simulation 1
Running Task 3 simulation 2
Running Task 3 simulation 3
Running Task 3 simulation 4
Running Task 3 simulation 5
Running Task 3 simulation 6
Task 3 complete!
